# 5 - CIDER Score vs TMB

**Purpose**: Scatter plots showing the weak correlation between CIDER Raw Score
and TMB, supporting their orthogonality as biomarkers. Per-cohort supplement.

| | |
|---|---|
| **Inputs** | `CIDER_PROJ_DIR/cpi_ready.rds` |
| **Outputs** | `figs/tmb_vs_tcell.png`, `tmb_vs_tcell_depth_adj.png` |

In [1]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))
source(paste0(dirname(getwd()), "/helpers/themes.r"))

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

In [2]:
library(ggpubr)
library(patchwork)

# 0 - Read data

In [ ]:
ready <-
  readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  transmute(
    sampleId, gp, cohort_broad_go,
    cdr3_unique, cdr3_total, cider_score_adj, cider_score_depth_adj, tmbPerMb, purity, responder
  ) |>
  pivot_longer(
    cols      = c(cdr3_unique, cdr3_total),
    names_to  = "cdr3_type",
    values_to = "cdr3_count"
  ) |>
  mutate(
    cdr3_type  = gsub("cdr3_", "", cdr3_type),
    cdr3_count = log2(cdr3_count + 1),
    type       = factor(
      ifelse(cdr3_type == "unique", "Unique CDR3s", "Total CDR3 Reads"),
      levels = c("Unique CDR3s", "Total CDR3 Reads")
    )
  )

# 1 - Main figure: TMB vs CIDER T-cell Score (Purity Adjusted)

In [ ]:
# Primary figure: TMB vs CIDER T-cell Score (purity-adjusted)
tmb_vs_tcell_adj <- ready |>
  filter(gp == "T-cell", cdr3_type == "unique") |>
  ggplot(aes(x = cider_score_adj, y = tmbPerMb)) +
  geom_point(alpha = 0.5, size = 0.9) + go_theme +
  geom_smooth(method = "lm") +
  scale_y_continuous(trans = log10p1_trans) +
  labs(
    y     = "TMB",
    x     = "CIDER T-cell Score (Purity Adjusted)",
    title = "TMB vs CIDER T-cell Score (Purity Adjusted)"
  ) +
  tags_main

set_plot_dims(5, 4.5); tmb_vs_tcell_adj

# 2 - Supporting: T-cell subcompartment

In [ ]:
# Primary scatter: CIDER T-cell Score (depth-adjusted) vs TMB
tmb_vs_tcell <- ready |>
  filter(gp == "T-cell", cdr3_type == "unique") |>
  ggplot(aes(x = cider_score_depth_adj, y = tmbPerMb)) +
  geom_point(alpha = 0.5, size = 0.9) +
  # Dark charcoal grey rather than ggplot's default blue -- keeps the fitted
  # line visually neutral against the categorical palette used elsewhere in
  # the paper (same convention as 14_depth_stability.ipynb's trend lines).
  geom_smooth(method = "lm", colour = "grey20") +
  scale_x_continuous(trans = log10p1_trans) +
  scale_y_continuous(trans = log10p1_trans) +
  go_theme +
  labs(
    x     = "CIDER T-cell Score",
    y     = "TMB (mut/Mb)",
    title = "TMB vs CIDER T-cell Score"
  ) +
  tags_main

set_plot_dims(5, 4.5); tmb_vs_tcell
saveRDS(tmb_vs_tcell, paste0(CIDER_PROJ_DIR, "figs/tmb_vs_tcell.rds"))

# 5 - Supplement: TMB vs CIDER Score

Reproduces Figure 5A using `cider_score_depth_adj` for direct comparison
with the purity-corrected version.

In [ ]:
# Sensitivity analysis: purity-corrected score for Figure S6A
tmb_vs_tcell_depth_adj <- ready |>
  filter(gp == "T-cell", cdr3_type == "unique") |>
  ggplot(aes(x = cider_score_adj, y = tmbPerMb)) +
  geom_point(alpha = 0.5, size = 0.9) +
  geom_smooth(method = "lm") +
  scale_x_continuous(trans = log10p1_trans) +
  scale_y_continuous(trans = log10p1_trans) +
  stat_cor(method = "pearson", size = 3.2) +
  go_theme +
  labs(
    x     = "CIDER T-cell Score (Purity Adjusted)",
    y     = "TMB (mut/Mb)",
    title = "TMB vs CIDER T-cell Score (Purity Adjusted)"
  ) +
  tags_supp

saveRDS(tmb_vs_tcell_depth_adj, paste0(CIDER_PROJ_DIR, "figs/tmb_vs_tcell_depth_adj.rds"))
set_plot_dims(5, 4.5); tmb_vs_tcell_depth_adj